# NB-08-01 — Tokenisation and Features

Deterministic preprocessing for English and Urdu, then bag-of-words, n-grams and TF-IDF — built by hand
first, then with scikit-learn. All corpora are synthetic and included here, so nothing is downloaded.

In [1]:
import math, re, unicodedata
from collections import Counter
print("ready")

ready


## 1. Normalisation and script detection

NFKC first, then decide about script variants. The Urdu corpus below is synthetic, written for this
course.

In [2]:
URDU = ["میں نے ادائیگی کی ہے لیکن رسید نہیں آئی۔ دوبارہ بھیجیں۔",
        "پارسل وقت پر نہیں پہنچا، ٹریکنگ صفحہ خالی ہے۔"]
ENGLISH = ["The parcel has not arrived; the tracking page is blank.",
           "Payment done, but no receipt yet."]

def has_urdu(text):
    return bool(re.search(r"[\u0600-\u06FF]", text))

def normalise(text):
    return unicodedata.normalize("NFKC", text).strip()

def tokenise(text):
    text = normalise(text)
    if has_urdu(text):
        return re.findall(r"[\u0600-\u06FF]+|[A-Za-z]+|\d+(?:\.\d+)?", text)
    return re.findall(r"[A-Za-z]+(?:'[a-z]+)?|\d+(?:\.\d+)?", text.lower())

def sentences(text):
    pattern = re.compile(r"[۔.!?]") if has_urdu(text) else re.compile(r"[.!?]")
    return [s.strip() for s in pattern.split(text) if s.strip()]

print(tokenise(ENGLISH[0]))
print(tokenise(URDU[1]))
print("urdu sentences:", len(sentences(URDU[0])))
print("negation kept:", "not" in tokenise(ENGLISH[0]), "نہیں" in tokenise(URDU[0]))

['the', 'parcel', 'has', 'not', 'arrived', 'the', 'tracking', 'page', 'is', 'blank']
['پارسل', 'وقت', 'پر', 'نہیں', 'پہنچا،', 'ٹریکنگ', 'صفحہ', 'خالی', 'ہے۔']
urdu sentences: 2
negation kept: True True


## 2. Bag of words by hand

Counts over a fixed vocabulary; note that order disappears.

In [3]:
DOCS = ["refund refund delay", "refund policy", "delay delay delay delay"]
vocab = sorted({w for d in DOCS for w in d.split()})
rows = [{w: d.split().count(w) for w in vocab} for d in DOCS]
print("vocab:", vocab)
for r in rows:
    print(r)

vocab: ['delay', 'policy', 'refund']
{'delay': 1, 'policy': 0, 'refund': 2}
{'delay': 0, 'policy': 1, 'refund': 1}
{'delay': 4, 'policy': 0, 'refund': 0}


## 3. N-grams

Bigrams keep negation. Vocabulary grows; that is the trade.

In [4]:
def ngrams(tokens, n):
    return ["_".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

texts = ["the service was not good", "the service was good"]
for t in texts:
    toks = tokenise(t)
    print(t, "->", ngrams(toks, 2))

uni = {w for t in texts for w in tokenise(t)}
bi = {g for t in texts for g in ngrams(tokenise(t), 2)}
print("unigram vocab:", len(uni), "bigram vocab:", len(bi))

the service was not good -> ['the_service', 'service_was', 'was_not', 'not_good']
the service was good -> ['the_service', 'service_was', 'was_good']
unigram vocab: 5 bigram vocab: 5


## 4. TF-IDF by hand

Three documents, the arithmetic shown in L08-03 section 4.

In [5]:
def tfidf(docs):
    v = sorted({w for d in docs for w in d.split()})
    n = len(docs)
    df = {w: sum(1 for d in docs if w in d.split()) for w in v}
    out = []
    for d in docs:
        counts = {w: d.split().count(w) for w in v}
        vec = {w: counts[w] * (math.log((n + 1) / (df[w] + 1)) + 1) for w in v}
        norm = math.sqrt(sum(x * x for x in vec.values())) or 1.0
        out.append({w: round(x / norm, 3) for w, x in vec.items()})
    return v, out

v, vecs = tfidf(DOCS)
for vec in vecs:
    print(vec)
print("cos(d1,d2) =", round(sum(vecs[0][w] * vecs[1][w] for w in v), 3))

{'delay': 0.447, 'policy': 0.0, 'refund': 0.894}
{'delay': 0.0, 'policy': 0.796, 'refund': 0.605}
{'delay': 1.0, 'policy': 0.0, 'refund': 0.0}
cos(d1,d2) = 0.541


## 5. The same thing with scikit-learn

Fit on train only; transform test with the fitted vocabulary.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

train = ["refund refund delay", "refund policy", "delay delay delay delay"]
test = ["refund refund refund"]

vec = TfidfVectorizer(sublinear_tf=True, norm="l2")
X_train = vec.fit_transform(train)
X_test = vec.transform(test)
print("train shape:", X_train.shape, "test shape:", X_test.shape)
print("features:", vec.get_feature_names_out())
print("sparsity:", round(1 - X_train.nnz / (X_train.shape[0] * X_train.shape[1]), 3))
print("test row:", X_test.toarray().round(3))

train shape: (3, 3) test shape: (1, 3)
features: ['delay' 'policy' 'refund']
sparsity: 0.444
test row: [[0. 0. 1.]]


## 6. Determinism

The same text must give the same vector, twice and after a fresh fit on the same training data.

In [7]:
a = TfidfVectorizer(sublinear_tf=True).fit(train).transform(test).toarray()
b = TfidfVectorizer(sublinear_tf=True).fit(train).transform(test).toarray()
print("PASS  deterministic" if (a == b).all() else "FAIL  deterministic")
print("PASS  negation preserved" if "not" in tokenise("not good") else "FAIL  negation")
print("PASS  idempotent" if normalise(normalise(" x ")) == normalise(" x ") else "FAIL  idempotent")

PASS  deterministic
PASS  negation preserved
PASS  idempotent


## What you should take away

- Normalisation and tokenisation are decisions; each destroys something, so log them.
- Urdu needs its own punctuation and its own token pattern, and no download is required to do it.
- Bigrams recover negation at the cost of vocabulary.
- TF-IDF by hand and via scikit-learn agree; fit on train only.
- Determinism is testable, and worth testing — silent refitting was one of the three postmortem causes.